# Himalia: from data to reusable visualization

This notebook demonstrates automatic plotting, a natural-language request, refinement, and exporting a reusable function. It uses synthetic data.

**Before running:** install Himalia into this notebook's Python environment (from the repository: `python -m pip install -e '.[dev,plotly]'`). Set `HIMALIA_MODEL` and your provider's API key in the environment that launches Jupyter, or use the configuration cell below.

Each `plot`, `fit`, or `refine` call uses your model API and may make one repair request. `render` and `save` do not use the API. Himalia sends a bounded summary/sample to your provider and runs generated Python locally; its checks are not a security sandbox.

In [ ]:
import os

import numpy as np
import pandas as pd

from himalia import Visualizer, plot

# You can instead set these in the shell before starting Jupyter.
# os.environ['HIMALIA_MODEL'] = 'openai/YOUR_MODEL_ID'
# from getpass import getpass
# os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

# Anthropic: use anthropic/YOUR_MODEL_ID and ANTHROPIC_API_KEY.
# Local Ollama: use ollama_chat/YOUR_INSTALLED_MODEL and
# os.environ['HIMALIA_API_BASE'] = 'http://localhost:11434'

if not os.getenv("HIMALIA_MODEL"):
    raise RuntimeError("Set HIMALIA_MODEL and your provider credentials before continuing.")

## 1. Automatic visualization of cross-validation results

Here each model has two metrics measured across five folds. Himalia sees the nested structure and chooses a visualization. The actual plotting function receives all the local data.

In [ ]:
results_dict = {
    "Ridge": {
        "r2": np.array([0.71, 0.75, 0.73, 0.70, 0.74]),
        "rmse": np.array([12.1, 11.3, 11.8, 12.4, 11.6]),
    },
    "Random forest": {
        "r2": np.array([0.80, 0.82, 0.79, 0.84, 0.81]),
        "rmse": np.array([9.7, 9.1, 10.0, 8.6, 9.4]),
    },
    "Gradient boosting": {
        "r2": np.array([0.83, 0.86, 0.82, 0.85, 0.84]),
        "rmse": np.array([8.9, 8.1, 9.2, 8.4, 8.7]),
    },
}

viz = plot(results_dict, backend="auto")

In [ ]:
print(viz.explanation)
# Optional: inspect exactly what was sent as the data profile.
# viz.profile

## 2. Refine the same visualization

A follow-up instruction uses the current function and the original data snapshot. You can inspect the resulting Python in `viz.code`.

In [ ]:
viz.refine(
    "Use one horizontal panel per metric. Show individual fold scores and a mean marker. "
    "Sort models by mean R², highest first, consistently across both panels."
)

In [ ]:
print(viz.code)

## 3. Reuse and export — no further inference

Use `render` to run the function locally on new data with the same structure. Use `save` to export a standalone Python function. Matplotlib figures can also be saved with `viz.figure.savefig(...)`.

In [ ]:
updated_results = {
    model: {metric: values * 0.98 for metric, values in metrics.items()}
    for model, metrics in results_dict.items()
}
viz.render(updated_results, title="Updated cross-validation scores")

In [ ]:
from pathlib import Path

# Choose a fresh filename on reruns so we do not overwrite existing code.
export_path = Path("vis_utils.py")
version = 2
while export_path.exists():
    export_path = Path(f"vis_utils_{version}.py")
    version += 1

viz.save(export_path, function_name="plot_cv_results")

In [ ]:
# The printed snippet works for normal imports. Here we load the chosen file
# directly so this cell also works when a rerun selected a numbered filename.
import importlib.util

spec = importlib.util.spec_from_file_location(export_path.stem, export_path)
vis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis_utils)

fig = vis_utils.plot_cv_results(updated_results, figsize=(11, 5))
fig

## 4. Describe a DataFrame visualization

`backend` selects the plotting library. `model` selects the LLM independently. The class-based API supports the same options as `plot`.

In [ ]:
rng = np.random.default_rng(42)
spend = rng.uniform(100, 1000, 120)
campaigns = pd.DataFrame(
    {
        "spend": spend,
        "revenue": 2.5 * spend + rng.normal(0, 180, len(spend)),
        "channel": rng.choice(["Search", "Social", "Email"], len(spend)),
    }
)

campaign_viz = Visualizer(backend="seaborn")
campaign_viz.fit(
    campaigns,
    prompt="Show the relationship between spend and revenue, colored by channel. "
    "Both monetary values are in euros. Make the chart clear and presentation-ready.",
)

## 5. Optional: interactive Plotly output

Install the Plotly extra and set the toggle below to `True` for another model request. The returned figure is interactive and supports `interactive_viz.figure.write_html('campaigns.html')`.

In [ ]:
RUN_PLOTLY = False

if RUN_PLOTLY:
    interactive_viz = plot(
        campaigns,
        prompt="Scatter plot of spend versus revenue, colored by channel, with hover labels.",
        backend="plotly",
    )